# 05. TaskGroup and timeouts

**Tier:** Deep. **Why you'd use this:** Several async jobs should start together, fail together, and respect a deadline. `TaskGroup` and `asyncio.timeout` give you that structure.

**After this lesson you can:** run coroutines in a `TaskGroup`; cancel work with `asyncio.timeout`; explain that a failure cancels siblings.
About 30 minutes. **Uses:** [Async](03-async-lesson.ipynb)


## Quick review

1. Why can unlocked `+=` lose updates across threads?
2. What does a `Lock` protect?

<details><summary>Answers</summary>

Read–modify–write is not one atomic step. The critical section you wrap so only one thread runs it at a time.
</details>


## The idea

`asyncio.TaskGroup` starts child tasks and waits for all of them. If one fails, the group cancels the others and raises an exception group. `asyncio.timeout` cancels the block if it runs too long. Together they give structured concurrency: lifetimes nest, and timeouts cannot silently leave orphan tasks.

Fire-and-forget tasks are easy to start and hard to reason about. Structured APIs force you to decide what "done" means before the block exits. Timeouts belong next to the work they limit so a slow dependency cannot hang a request forever. Combine both when a request fans out to several backends.

<details><summary>Explain more</summary>

Structured concurrency means tasks started inside a block finish or cancel before the block ends. That beats fire-and-forget `create_task` scattered across a function.
</details>


## See it work

Fetch with a timeout, then show sibling cancellation on failure.


In [1]:
import asyncio


async def fetch(name, delay):
    # 1. Simulate a slow dependency.
    await asyncio.sleep(delay)
    return name


async def bounded():
    # 2. timeout cancels if the block runs too long.
    try:
        async with asyncio.timeout(0.05):
            return await fetch("ok", 0.01)
    except TimeoutError:
        return "timed out"


print(await bounded())


async def one_fails():
    async def boom():
        await asyncio.sleep(0.01)
        raise RuntimeError("nope")

    async def slow():
        await asyncio.sleep(1)
        return "late"

    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(boom())
            late = group.create_task(slow())
    except Exception as error:
        return type(error).__name__, late.cancelled()
    return "ok", False


print(await one_fails())

ok
('ExceptionGroup', True)


## Mental model

Structured concurrency is a meeting that cannot adjourn until every agenda item finishes or is cancelled. `TaskGroup` is the meeting room. `asyncio.timeout` is the wall clock that ends the discussion. Leaving a task running after you return is like adjourning while someone still has the microphone—errors show up later as "never retrieved."

| tool | guarantees |
|---|---|
| `TaskGroup` | children done or cancelled before exit |
| failure in one child | siblings cancelled; exception group raised |
| `asyncio.timeout` | block cancelled after deadline |
| bare `create_task` | no lifetime owner unless you await |

Use groups around fan-out for a single request. Use timeouts on the outer boundary so nested waits cannot exceed the client's patience. Inspect `ExceptionGroup.exceptions` when you need per-child recovery. Prefer failing fast for user-facing requests; prefer gathering partial results only when the product explicitly allows degraded answers.

When migrating older `gather` code, keep gather for simple ordered fan-in with no cancellation story, and switch to TaskGroup when a failure should abort the whole set. Timeouts on the outside prevent a single slow child from ignoring a parent deadline.

Deadlines and ownership travel together: the timeout wraps the group when the client sets a single budget for the whole fan-out. Putting a timeout on each child without an outer budget can still overrun if you start children sequentially by mistake. Keep the outer shape obvious in code review.


## Check yourself

Answer these from the example above before you run the cell.


1. What happens to sibling tasks when one TaskGroup child raises?
2. Which exception does `asyncio.timeout` raise on deadline?
3. Why must cancellable code hit `await` points?

<details><summary>Answers</summary>

1. They are cancelled, then an exception group is raised. 2. `TimeoutError`. 3. Cancellation is delivered at await boundaries.
</details>


In [2]:
import asyncio


async def work(label, delay):
    await asyncio.sleep(delay)
    return label


async def main():
    async with asyncio.TaskGroup() as group:
        first = group.create_task(work("a", 0.01))
        second = group.create_task(work("b", 0.02))
    return first.result(), second.result()


print(await main())

('a', 'b')


<details><summary>Check your answers</summary>

`('a', 'b')`. The `async with` waits until both tasks finish.
</details>


## Why it works this way

`TaskGroup` registers child tasks and, on exit, waits for them. On child failure it cancels siblings, then raises `ExceptionGroup` (or a subclass) so callers see every error. That nesting matches how `with` nests resource lifetimes. `asyncio.timeout` sets a deadline on the current task; exceeding it raises `TimeoutError` and cancels the work inside the block. Deep dive: pair with the async lesson—the loop still runs everything cooperatively.

Cost: task objects and cancellation checkpoints at `await`. Code that never awaits cannot be cancelled promptly. Busy CPU inside a coroutine delays timeout handling until the next await. The design reason is safer concurrency than manual `create_task` lists you might forget to await. Prefer TaskGroup when children share a lifetime; prefer `gather` when you only need ordered results and a simpler error policy (`return_exceptions` vs fail-fast).

Evidence: TaskGroup returns both results; a tight timeout still allows a fast fetch; a failing child cancels the slow sibling (`cancelled()` true). Trace:

| step | boom | slow | group |
|---|---|---|---|
| 1 | runs | runs | open |
| 2 | raises | cancelled | closing |
| 3 | | | raises ExceptionGroup |

Orphan tasks that raise after the parent returned often log "task exception was never retrieved." Owning them in a group surfaces the error at the `async with` boundary.

Timeout and cancel checks print booleans from `cancelled()` and caught errors. When a wall-clock bound is part of the claim, that cell is tagged `volatile` so CI tolerates slow hosts.

Cancellation and locking both exist to make concurrent code predictable under interleaving; without them, demos flake and production fails silently.


In [3]:
import asyncio


async def demo():
    async with asyncio.timeout(0.05):
        await asyncio.sleep(0.01)
        return "fast"


print(await demo())


async def cancel_demo():
    async def boom():
        raise RuntimeError("x")

    async def slow():
        try:
            await asyncio.sleep(1)
        except asyncio.CancelledError:
            return "cancelled"
        return "done"

    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(boom())
            task = group.create_task(slow())
    except Exception:
        await asyncio.sleep(0)
        return task.cancelled() or task.result() == "cancelled"
    return False


print(await cancel_demo())

fast
True


## When to use it

Use `TaskGroup` for structured sets of tasks. Use `asyncio.timeout` for deadlines on a block. Use `gather` for simpler ordered fan-in without cancellation plumbing. Don't leave background tasks without an owner. Don't catch `ExceptionGroup` unless you intend to handle multiple failures.

Use TaskGroup for request-scoped fan-out. Use timeout around the slowest acceptable wait. Use gather for simple ordered collect. Do not swallow ExceptionGroup without inspecting `exceptions`. Do not create tasks you never await or group.


## Common mistake

Starting tasks with `asyncio.create_task` and returning before they finish, so errors surface as "task exception was never retrieved."


In [4]:
import asyncio


async def leak():
    async def boom():
        raise RuntimeError("lost")

    asyncio.create_task(boom())
    await asyncio.sleep(0.01)
    return "returned anyway"


print(await leak())

returned anyway


Own the task in a `TaskGroup` or await it explicitly so failures propagate.


In [5]:
import asyncio


async def owned():
    async def boom():
        raise RuntimeError("lost")

    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(boom())
    except Exception as error:
        return type(error).__name__
    return "ok"


print(await owned())

ExceptionGroup


## Try it

Change `bounded` so the fetch sleeps `0.1` seconds. What does the function return, and which context manager decided that?


## Recap

- `TaskGroup` owns child task lifetimes.
- Failures cancel siblings and raise an exception group.
- `asyncio.timeout` enforces a deadline on a block.

**Next:** [practice](05-taskgroup-timeouts-practice.ipynb)
